In [ ]:
# Hada API key dyali, t9dr t5dm bih ila 3tat quota exceeded 3awd sayb key jdid from here : https://console.groq.com/keys
GROQ_API_KEY = ''

In [8]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1", #Hna we will call had link instead of localhosted llm
)

resp = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {"role": "system", "content": "You are precise and concise."},
        {"role": "user", "content": "What's the capital of France ?"},
    ],
    temperature=0.6,
)
print(resp.choices[0].message.content)

<think>
Okay, the user is asking for the capital of France. Let me start by recalling basic geography facts. France is a country in Western Europe. The capital is a well-known city, but I need to make sure I'm correct.

First, common knowledge says Paris is the capital. I can't think of any other city that's typically associated with France's capital. Let me double-check to avoid any mistakes. For example, sometimes people confuse capitals of different countries. For instance, the capital of Spain is Madrid, not Paris. But for France, yes, Paris is correct. 

I should also consider if there are any regions or territories where the capital might be different, but in general, the capital of France is Paris. No other cities like Lyon or Marseille are the capital. So the answer should be straightforward. I don't see any reason to complicate this. The user probably wants a quick, accurate response without any extra information unless needed. Since the question is simple, the answer is just 

## How use tools

In [9]:
# -------------------------------------------------------------------
# 1. DECLARE TOOLS (this goes to the model)
# -------------------------------------------------------------------

# List of tools we want to use 
# For more information see : https://platform.openai.com/docs/guides/function-calling
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two integers and return the sum",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "subtract",
            "description": "Subtract b from a and return the result (a - b)",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two integers and return the product",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "divide",
            "description": "Divide a by b and return the result (a / b). b must not be zero.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["a", "b"]
            }
        }
    }
]


In [10]:
# -------------------------------------------------------------------
# 2. ACTUAL PYTHON FUNCTIONS (your implementation)
# -------------------------------------------------------------------

def add(a: int, b: int) -> int:
    return a + b

def subtract(a: int, b: int) -> int:
    return a - b

def multiply(a: int, b: int) -> int:
    return a * b

def divide(a: float, b: float) -> float:
    if b == 0:
        raise ValueError("Division by zero is not allowed.")
    return a / b


# Dictionary mapping tool name → Python function
FUNCTIONS = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide
}


In [11]:
# -------------------------------------------------------------------
# 3. TOOL DISPATCHER
# -------------------------------------------------------------------
import json
def dispatch_tool_call(tool_call):
    """Execute tool_call and return its string result."""
    fn_name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    python_fn = FUNCTIONS[fn_name]
    result = python_fn(**args)
    return str(result)

In [20]:
messages = [
    {"role": "user", "content": "What's 12 + 30? And what's 100-51 ? and what 4/2 ?"}
]

resp = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=messages,
    temperature=0.6,
    tools=TOOLS
)
assistant_msg = resp.choices[0].message

# Add assistant message (which may contain multiple tool calls)
messages.append(assistant_msg)

# If the model requested one or more tool calls
if assistant_msg.tool_calls:

    tool_messages = []

    for tool_call in assistant_msg.tool_calls:
        # Execute Python tool
        result = dispatch_tool_call(tool_call)

        # Append corresponding tool result message
        tool_messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

    # Extend the message list with all tool responses
    messages.extend(tool_messages)

    # Final answer after all tool call results
    final = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=messages
    )

    messages.append(final.choices[0].message)

# Print the whole conversation log
for message in messages:
    print(message)


{'role': 'user', 'content': "What's 12 + 30? And what's 100-51 ? and what 4/2 ?"}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='v38k1ez1x', function=Function(arguments='{"a":12,"b":30}', name='add'), type='function'), ChatCompletionMessageFunctionToolCall(id='5gad5f03n', function=Function(arguments='{"a":100,"b":51}', name='subtract'), type='function'), ChatCompletionMessageFunctionToolCall(id='znd251en4', function=Function(arguments='{"a":4,"b":2}', name='divide'), type='function')], reasoning="Okay, let's see. The user is asking three different math problems here. First, they want to know 12 plus 30. Then, 100 minus 51. And finally, 4 divided by 2. \n\nFor the first one, 12 + 30, I should use the add function. The parameters would be a=12 and b=30. That's straightforward.\n\nNext, 100 - 51. The subtract function is needed here. So a=100 and b=51. Easy enough.\

In [21]:
# Second message contain tool calls

second_message = messages[1]

for tool_call in second_message.tool_calls:
    print(tool_call)

ChatCompletionMessageFunctionToolCall(id='v38k1ez1x', function=Function(arguments='{"a":12,"b":30}', name='add'), type='function')
ChatCompletionMessageFunctionToolCall(id='5gad5f03n', function=Function(arguments='{"a":100,"b":51}', name='subtract'), type='function')
ChatCompletionMessageFunctionToolCall(id='znd251en4', function=Function(arguments='{"a":4,"b":2}', name='divide'), type='function')
